In [2]:
# For now, we still need to manually add atomworks dependency
import sys;
sys.path.insert(0, '/home/jbutch/Projects/HT25/af3/rfd3-release/lib/atomworks/src')
sys.path.append('/home/raswanth/foundry/models/rfd3/src')
sys.path.append('/home/jbutch/Projects/HT25/af3/rfd3-release/src')

from atomworks.io.utils.visualize import view

<jemalloc>: arena 0 background thread creation failed (11)


# 1) Design with RFD3

In [4]:
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine
from rfd3.inference.input_parsing import DesignInputSpecification
conf = RFD3InferenceConfig(
    ckpt_path='/home/raswanth/.foundry/checkpoints/rfd3_latest.ckpt',
    diffusion_batch_size=2,
    dump_trajectories=True,
    low_memory_mode=True,
)
# Lazily initialize model
model = RFD3InferenceEngine(**conf)

19:49:35 DEBUG transforms: Debug mode is on
19:49:37 INFO rfd3.engine: [rank: 0] Low memory mode enabled.


In [5]:
spec = DesignInputSpecification(
        input='/home/raswanth/foundry/theozymes/Theozyme-Y301K-full-edited.pdb',
        length='380-420',
        ligand='L:G',
        unindex='A82,A104-106,A185,A228-231,A298-299,A301,A345,A371',
    )
# view(spec.build())

19:50:32 WARNING atomworks.io: We can't fix formal charges without building from templates, as we need to know the true number of hydrogens bonded to a given atom, not the inferred number. This may lead to occasional inaccuracies after adding inter-residue bonds. To avoid this and fix formal charges, set `add_missing_atoms = True`.
19:50:32 WARNING atomworks.io: Chain A contains both polymer and non-polymer residues; separating them for processing, naming the non-polymer residues as B.


In [5]:
outputs = model.run(
    inputs=spec,
    n_batches=10,
)
outputs.keys()

Using bfloat16 Automatic Mixed Precision (AMP)
You are using a CUDA device ('NVIDIA GeForce RTX 4090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
01:15:44 WARNING foundry.utils.weights: [rank: 0] Failed to apply policy: 'copy' to 'model.token_initializer.chunked_pairwise_embedder.motif_pos_embedder.output_proj.weight': Parameter 'model.token_initializer.chunked_pairwise_embedder.motif_pos_embedder.output_proj.weight' not found in checkpoint. Falling back to policy: 'reinit'.
01:15:44 WARNING foundry.utils.weights: [rank: 0] Failed to apply policy: 'copy' to 'model.token_initializer.chunked_pairwise_embedder.motif_pos_embedder.process_valid_mask.weight': Parameter 'model.token_initializer.chunked_pairwise_embedder.motif_pos_em

dict_keys(['backbone_0_4', 'backbone_0_1', 'backbone_0_7', 'backbone_0_5', 'backbone_0_3', 'backbone_0_9', 'backbone_0_0', 'backbone_0_8', 'backbone_0_6', 'backbone_0_2'])

In [9]:
for idx, data in outputs.items():
    print(f"Output type for batch {idx}: {type(data)}[0] = {type(data[0])}")
    print(f"Output atom_array: {data[0].atom_array}")
    atom_array = data[0].atom_array

# Extract the first generated backbone for downstream use
first_key = next(iter(outputs.keys()))
atom_array = outputs[first_key][0].atom_array

view(atom_array)

Output type for batch backbone_0_4: <class 'list'>[0] = <class 'rfd3.engine.RFD3Output'>
Output atom_array:     A       1  LEU N      N        -6.765   21.314   13.605
    A       1  LEU CA     C        -6.179   20.312   12.782
    A       1  LEU C      C        -7.153   19.793   11.717
    A       1  LEU O      O        -6.774   19.565   10.593
    A       1  LEU CB     C        -5.637   19.187   13.619
    A       1  LEU CG     C        -4.453   19.442   14.530
    A       1  LEU CD1    C        -4.170   18.319   15.414
    A       1  LEU CD2    C        -3.240   19.826   13.729
    A       2  LEU N      N        -8.423   19.624   12.097
    A       2  LEU CA     C        -9.462   19.190   11.154
    A       2  LEU C      C        -9.645   20.295   10.101
    A       2  LEU O      O        -9.839   19.999    8.952
    A       2  LEU CB     C       -10.781   18.960   11.851
    A       2  LEU CG     C       -10.844   17.581   12.590
    A       2  LEU CD1    C       -12.123   17.559  

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [10]:
from mpnn.inference_engines.mpnn import MPNNInferenceEngine

# Configure MPNN inference engine
# See mpnn.utils.inference.MPNN_GLOBAL_INFERENCE_DEFAULTS for all options
engine_config = {
    "model_type": "ligand_mpnn",  # or "protein_mpnn" for vanilla ProteinMPNN
    "is_legacy_weights": True,    # Required for now for ligand_mpnn and protein_mpnn
    "out_directory": "/home/raswanth/foundry/enzyme_output",        # Return results in memory
    "write_structures": True,
    "write_fasta": True,
}

# Configure per-input inference options
# See mpnn.utils.inference.MPNN_PER_INPUT_INFERENCE_DEFAULTS for all options
input_configs = [
    {
        "batch_size": 5,         # Generate 10 sequences per structure
        "remove_waters": True,
    }
]

# Run sequence design on the RFD3-generated backbone
model = MPNNInferenceEngine(**engine_config)
mpnn_outputs = model.run(input_dicts=input_configs, atom_arrays=[atom_array])

11:27:16 WARNING atomworks.io.utils.ccd: The following CCD codes were not found in the local mirror at : {np.str_('L:G')}


In [11]:
from biotite.structure import get_residue_starts
from biotite.sequence import ProteinSequence

# Extract and display the designed sequences
print(f"Generated {len(mpnn_outputs)} designed sequences:\n")

for i, item in enumerate(mpnn_outputs):
    res_starts = get_residue_starts(item.atom_array)
    # Convert 3-letter codes to 1-letter using Biotite
    seq_1letter = ''.join(
        ProteinSequence.convert_letter_3to1(res_name)
        for res_name in item.atom_array.res_name[res_starts] if res_name != 'L:G'
    )
    print(f"Sequence {i+1}: {seq_1letter}")

Generated 5 designed sequences:

Sequence 1: LIELSRAFHKENKMEGLGADEAKFRAAAAVMARFPNVVVWAAPAATEPEHQAVAAALARELAAAAGIPADRFRLDDAGYAALNGRLARQDRAGELTLLPLYGGTPSFTDEEVAAAAEQAERAVAESDPDRPVRIAISVGSAAAGFSGDNPTSDAMRLGLTTRIVRGLLAACGGRVALVVRASGVRGDLLSDRATIEIHGDADATLLVVPGGHTFAGALSALEAADATGLPRVVAALTSGAAMDPLGREAVEELRARGVEIVLRVRVAGGLGAVPAAIRRAARECADFAPVDILTISGTNIAGQSGFDAIIAAAAEAAATTGAALALISGGAMRRMTQEQLTALIAALKAAGVRSVIALVHSEEEAAKALAAGADGVVLQEVDLGVGGTSLLSPAVAAAKALGLDIQSLGLLSAGEAKSLP
Sequence 2: LIELSRAFRREHGREGHGADEAKVRAAAAVIARYPNVEVWAAPEATRPDHRAVARELARQLAAAAGIPADRFRFDDRGYAALSGELSRQDRAGRLRMLPLSGGTNNFTDAEVAAAAEQLEAAVAESDPATPVAISIGVGGPAAGFSGDNPTSDAMRLGLSTGVVRGLLAAAGGRVALVVEASGVPGDLLSDEAVIEIVGAPDADLLVVPGGHTWAGARTALAAHEATGLPRTVACVTNGAFTDPLGREALAELKARGITIVGAARVAGGLGATPAAIRRGAELGAAFGPVDVLVISGINVVGQSGFDALIAAAAEAARTTGAPLALIAGGAMERMTQEQLAALIAALKAAGVRSVVANVHSDELAAKALAAGADGVVKFEVDLGFGGSSLLLPAVAAAKELGLDLQSLGLLSAGRIASLP
Sequence 3: LIALSRAFHAENQMEGLGADEAQFRATARVMAKYPHVIAWAAPEATRPEHQEVAEALARELAAAAGIPADRFRLDRRGYAALSGELARL

In [12]:
from rf3.inference_engines.rf3 import RF3InferenceEngine
from rf3.utils.inference import InferenceInput


# Initialize RF3 inference engine
inference_engine = RF3InferenceEngine(ckpt_path='/home/raswanth/.foundry/checkpoints/rf3_foundry_01_24_latest_remapped.ckpt', verbose=False)

# Create input from the MPNN-designed structure (first design)
# This re-folds the sequence to validate it adopts the intended structure
input_structure = InferenceInput.from_atom_array(atom_array, example_id="example_protein")
rf3_outputs = inference_engine.run(inputs=input_structure)

# Outputs: dict mapping example_id -> list[RF3Output] (multiple models per input)
print(f"Output keys: {rf3_outputs.keys()}")
print(f"Number of models for 'example_protein': {len(rf3_outputs['example_protein'])}")

11:27:33 WARNING atomworks.io: The `extra_fields` argument will be ignored if there is no CIF file input.
11:27:33 WARNING atomworks.io: Adding missing atoms will erase extra fields. If you just want to load a structure with the given extra fields, you should probably use the much faster 'load_any' function from atomworks.io.utils.io_utils instead of 'parse'. Parse is meant for cleaning up structures from the RCSB PDB.
11:27:33 WARNING atomworks.io.utils.ccd: The following CCD codes were not found in the local mirror at : {np.str_('L:G')}
11:27:33 INFO rf3.inference_engines.rf3: [rank: 0] Loading checkpoint from /home/raswanth/.foundry/checkpoints/rf3_foundry_01_24_latest_remapped.ckpt...


11:27:33 WARNING atomworks.ml: Using element type for atom names of atomized tokens.
Using bfloat16 Automatic Mixed Precision (AMP)
11:27:35 WARNING rf3.inference_engines.rf3: [rank: 0] out_dir is None - results will be returned in memory! If you want to save to disk, please provide an out_dir.
11:27:35 INFO rf3.inference_engines.rf3: [rank: 0] Found 1 structures to predict!
11:27:35 INFO rf3.inference_engines.rf3: [rank: 0] Predicting structure 1/1: example_protein
11:27:35 WARNING atomworks.ml: Cached data not found for ALA at /net/tukwila/lschaaf/datahub/MACE-OMOL-Jul2025/mace_embeddings/A/ALA/ALA.pt
11:27:35 WARNING atomworks.ml: Cached data not found for ARG at /net/tukwila/lschaaf/datahub/MACE-OMOL-Jul2025/mace_embeddings/A/ARG/ARG.pt
11:27:35 WARNING atomworks.ml: Cached data not found for ASN at /net/tukwila/lschaaf/datahub/MACE-OMOL-Jul2025/mace_embeddings/A/ASN/ASN.pt
11:27:35 WARNING atomworks.ml: Cached data not found for ASP at /net/tukwila/lschaaf/datahub/MACE-OMOL-Jul202

Output keys: dict_keys(['example_protein'])
Number of models for 'example_protein': 5


In [13]:
# Extract the top-ranked prediction
rf3_output = rf3_outputs["example_protein"][0]

# Inspect RF3Output structure
print(f"RF3Output contains:")
print(f"  - atom_array: {len(rf3_output.atom_array)} atoms")
print(f"  - summary_confidences: {list(rf3_output.summary_confidences.keys())}")
print(f"  - confidences: {list(rf3_output.confidences.keys()) if rf3_output.confidences else None}")

# Visualize the predicted structure
view(rf3_output.atom_array)

RF3Output contains:
  - atom_array: 2681 atoms
  - summary_confidences: ['chain_ptm', 'chain_pair_pae_min', 'chain_pair_pde_min', 'chain_pair_pae', 'chain_pair_pde', 'overall_plddt', 'overall_pde', 'overall_pae', 'ptm', 'iptm', 'has_clash', 'ranking_score']
  - confidences: ['atom_chain_ids', 'atom_plddts', 'pae', 'token_chain_ids', 'token_res_ids']


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [14]:
# Summary confidences: overall model quality metrics
summary = rf3_output.summary_confidences

print("=== Summary Confidences ===")
print(f"  Overall pLDDT:    {summary['overall_plddt']:.3f}")
print(f"  Overall PAE:      {summary['overall_pae']:.2f} A")
print(f"  Overall PDE:      {summary['overall_pde']:.3f}")
print(f"  pTM:              {summary['ptm']:.3f}")
print(f"  ipTM:             {summary.get('iptm', 'N/A (single chain)')}")
print(f"  Ranking score:    {summary['ranking_score']:.3f}")
print(f"  Has clash:        {summary['has_clash']}")

=== Summary Confidences ===
  Overall pLDDT:    0.717
  Overall PAE:      18.00 A
  Overall PDE:      6.740
  pTM:              0.511
  ipTM:             0.3853078782558441
  Ranking score:    0.410
  Has clash:        False


In [15]:
# Detailed per-atom/residue confidences
conf = rf3_output.confidences

print("=== Per-Atom/Residue Confidences ===")
print(f"  atom_plddts:      {len(conf['atom_plddts'])} values (one per atom)")
print(f"  atom_chain_ids:   {len(conf['atom_chain_ids'])} values")
print(f"  token_chain_ids:  {len(conf['token_chain_ids'])} values (one per residue)")
print(f"  token_res_ids:    {len(conf['token_res_ids'])} values")
print(f"  PAE matrix:       {len(conf['pae'])}x{len(conf['pae'][0])}")

# Preview first 10 atom pLDDT scores
import numpy as np
print(f"\nFirst 10 atom pLDDTs: {np.round(conf['atom_plddts'][:10], 2).tolist()}")

=== Per-Atom/Residue Confidences ===
  atom_plddts:      2681 values (one per atom)
  atom_chain_ids:   2681 values
  token_chain_ids:  442 values (one per residue)
  token_res_ids:    442 values
  PAE matrix:       442x442

First 10 atom pLDDTs: [0.63, 0.64, 0.65, 0.61, 0.63, 0.61, 0.6, 0.55, 0.65, 0.66]


In [16]:
from biotite.structure import rmsd, superimpose
from atomworks.constants import PROTEIN_BACKBONE_ATOM_NAMES
import numpy as np

# Get structures for comparison
aa_generated = atom_array              # Original RFD3 backbone (from Section 1)
aa_refolded = rf3_output.atom_array    # RF3-predicted structure

# Filter to backbone atoms (N, CA, C, O)
bb_generated = aa_generated[np.isin(aa_generated.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)]
bb_refolded = aa_refolded[np.isin(aa_refolded.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)]

# Superimpose structures and calculate RMSD
bb_refolded_fitted, _ = superimpose(bb_generated, bb_refolded)
rmsd_value = rmsd(bb_generated, bb_refolded_fitted)

print(f"Backbone RMSD: {rmsd_value:.2f} A")
print(f"\nInterpretation: {'Excellent' if rmsd_value < 1.0 else 'Good' if rmsd_value < 2.0 else 'Moderate'} designability")

Backbone RMSD: 17.08 A

Interpretation: Moderate designability


In [17]:
from atomworks.io.utils.io_utils import to_cif_file

# Export structures to CIF format for visualization in PyMOL/ChimeraX
to_cif_file(aa_generated, "generated.cif")
to_cif_file(aa_refolded, "refolded.cif")

print("Exported structures:")
print("  - generated.cif: Original RFD3 backbone")
print("  - refolded.cif:  RF3-predicted structure")

Exported structures:
  - generated.cif: Original RFD3 backbone
  - refolded.cif:  RF3-predicted structure
